Example Phase 1 Data Collection:

In this phase i either directly downloaded the csv files from Kaggle or used the kagglehub library to download the files. Some datasets where just available in tsv files so i had to transform them to csv files. 
Firstly we need to establish a connection to kaggle with the kagglehub library, after that we can use the dataset downloading and preprocesssing class to pre clean the data from kaggle
The example Data i take here will be from the Dataset that we used to generate our Data for the categories , positive, normal and negative. 

In [39]:
import pandas as pd
import kagglehub
import os

if not os.path.exists('./datasets/amazon_reviews_us_Software_v1_00jupyternotebook.tsv'):

 download = kagglehub.dataset_download(
     'cynthiarempel/amazon-us-customer-reviews-dataset', path ='amazon_reviews_us_Software_v1_00.tsv', output_dir= './datasets/raw_data'
    )
 
 tsv_file = './datasets/raw_data/amazon_reviews_us_Software_v1_00.tsv'
 csv_table = pd.read_table(tsv_file,sep='\t', on_bad_lines='warn')
 display(csv_table.head(3))  # showing the initial and original csv
 csv_table.to_csv('./jupyter_notebook_presentation/jupyter_notebook_phase_2/amazon_reviews_us_Software_v1_00jupyter.csv', index= False)


/var/folders/8m/hcy33_r91zv_cx2w_75k9bsm0000gn/T/ipykernel_54795/2187677960.py:12: ParserWarning: Skipping line 8021: expected 15 fields, saw 22
Skipping line 34886: expected 15 fields, saw 22
Skipping line 49286: expected 15 fields, saw 22

  csv_table = pd.read_table(tsv_file,sep='\t', on_bad_lines='warn')


,marketplace,customer_id,review_id,product_id,product_parent,product_title,product_category,star_rating,helpful_votes,total_votes,vine,verified_purchase,review_headline,review_body,review_date
0,US,42605767,R3EFW2STIYIY0I,B00MUTIDKI,248732228,McAfee 2015 Internet Security 3 PC (3-Users),Software,1,2,2,N,Y,I was very disappointed with this,I was very disappointed with this. The descrip...,2015-08-31
1,US,51771800,R12NR0R5A9F7FT,B00EPACNUG,531462352,Hallmark Card Studio 2014,Software,5,0,0,N,Y,Five Stars,"I had a little struggle getting familiarized, ...",2015-08-31
2,US,16053526,R1LSH74R9XAP59,B00164AZA4,473982505,Search and Rescue 4,Software,2,0,1,N,Y,Have windows 10?,Tried to download it on my Windows 10 and it w...,2015-08-31


First cleaning of the transformed csv-file, the string will have a minimum and maximum length to provide higher quality data. 

In [40]:


df_sentiment = pd.read_csv('./jupyter_notebook_presentation/jupyter_notebook_phase_2/amazon_reviews_us_Software_v1_00jupyter.csv')
df_sentiment = df_sentiment.loc[df_sentiment['review_body'].str.len()>25]
df_sentiment = df_sentiment.loc[df_sentiment['review_body'].str.len()<250]
display(df_sentiment.head(3)) 



,marketplace,customer_id,review_id,product_id,product_parent,product_title,product_category,star_rating,helpful_votes,total_votes,vine,verified_purchase,review_headline,review_body,review_date
1,US,51771800,R12NR0R5A9F7FT,B00EPACNUG,531462352,Hallmark Card Studio 2014,Software,5,0,0,N,Y,Five Stars,"I had a little struggle getting familiarized, ...",2015-08-31
2,US,16053526,R1LSH74R9XAP59,B00164AZA4,473982505,Search and Rescue 4,Software,2,0,1,N,Y,Have windows 10?,Tried to download it on my Windows 10 and it w...,2015-08-31
3,US,15319481,R1QXUNTF76K7L6,B00E6LIEFM,189774198,Quickbooks Pro,Software,2,0,0,N,Y,"Disc was corrupt, had to spend a couple hours ...","Disc was corrupt, had to spend a couple hours ...",2015-08-31


Dropping all of the unnecessary columns, we dont need these.

In [ ]:

df_sentiment = df_sentiment.drop(columns = ['marketplace','customer_id','review_id','product_id','product_parent','product_title','product_category','helpful_votes','total_votes','vine','verified_purchase','review_headline','review_date'])
display(df_sentiment.head(3)) 

,star_rating,review_body
1,5,"I had a little struggle getting familiarized, ..."
2,2,Tried to download it on my Windows 10 and it w...
3,2,"Disc was corrupt, had to spend a couple hours ..."


The sentiment column is very important to add labels later, for the training of our bert model. That's why i added the column here, after a small but effective filter for the different sentiments.

In [42]:
df_sentiment.loc[df_sentiment['star_rating'].isin([4,5]) , 'sentiment'] = 'positive'
df_sentiment.loc[df_sentiment['star_rating'].isin ([3]) , 'sentiment'] = 'neutral'
df_sentiment.loc[df_sentiment['star_rating'].isin([1,2]) , 'sentiment'] = 'negative'
display(df_sentiment.head(3)) 

df_sentiment.to_csv("./jupyter_notebook_presentation/jupyter_notebook_phase_2/amazon_reviews_us_Software_v1_00jupyter.csv",index=False)

,star_rating,review_body,sentiment
1,5,"I had a little struggle getting familiarized, ...",positive
2,2,Tried to download it on my Windows 10 and it w...,negative
3,2,"Disc was corrupt, had to spend a couple hours ...",negative


The first step is now done, we have the data from Kaggle and we slightly cleaned them, now we need to apply further filters to get more meaningful data. 
As an example i will show how we filtered the positive reviews. 
For this we use the data_filter_collecting_and_appending class.

Loading the csv and shuffling it to achieve a higher data Quality

In [ ]:

df = pd.read_csv("./jupyter_notebook_presentation/jupyter_notebook_phase_2/amazon_reviews_us_Software_v1_00jupyter.csv")
dfshuffled = df.sample(frac=1)   
 
df_positiveReviews = dfshuffled 

display(df_positiveReviews.head(3)) 


,star_rating,review_body,sentiment
60665,5,Loved it for the price compared to the 2014 st...,positive
48841,5,Purchased several versions of turbo tax. Grea...,positive
90771,3,I had issues with having too many licences and...,neutral


Introducing a word pattern that is present in the most positive reviews.
Filtering with star_rating and word patterns, we need 1500 reviews that are positive for our training data thats why we use head(1500).
The star_rating column is irrelevant now, because we are looking firstly at the sentiment.

In [34]:

positivesummaryPatterns = "good|nice|awesome|perfect|excellent|great"

df_positiveReviews = df_positiveReviews.loc[(df_positiveReviews['star_rating'].isin([4,5])) & (df_positiveReviews['review_body'].str.contains(positivesummaryPatterns,na= False, case= False))].head(1500) 
df_positiveReviews = df_positiveReviews.drop(columns ='star_rating') 
display(df_positiveReviews.head(3))

,review_body,sentiment
48841,Purchased several versions of turbo tax. Grea...,positive
52972,I was told that the mac version of quickbooks ...,positive
28731,The book helped greatly with the side quest'...,positive


After the extraction of the positive reviews we remove them from the bigger dataset, this is a safety mechanism so a review that is already in a dataset will not be used in a other category even if we have different filters and we most likely will not mix these reviews we should take the neccesary steps to not use data twice under any circumstances. 
As we can see after the removal of the positive reviews from the whole dataset we have exactly 1500 reviews less than before.

In [ ]:

df_positiveReviews.to_csv("./jupyter_notebook_presentation/jupyter_notebook_phase_2/file_appending_method_demo/positive.csv", index_label="ID")

dfshuffled = dfshuffled.drop(df_positiveReviews.index)  
dfshuffled.to_csv("jupyter_notebook_presentation/jupyter_notebook_phase_2/bigger_dataset_after_removal_jupyter/DatasetafterPosRm.csv", index_label="ID") 

For the demonstration of the appending of our different csv category files we need to collect further csv Files wirh different categories. 
We need to collect them in a folder so we can append them. 
The review categories are negation and negative these also need their own filters.

In [ ]:
df_negation = pd.read_csv("jupyter_notebook_presentation/jupyter_notebook_phase_2/bigger_dataset_after_removal_jupyter/DatasetafterPosRm.csv", index_col=0) 
df_negationReviews = df_negation.copy()

negationsummaryPatterns = "No|Never|Not|wasn't"

df_negationReviews = df_negationReviews.loc[(df_negationReviews['star_rating'].isin([1,2,3])) & (df_negationReviews['review_body'].str.contains(negationsummaryPatterns,na= False, case=False))].head(1000) 
df_negationReviews["sentiment"] = df_negationReviews["sentiment"].replace("negative","negation")
df_negationReviews["sentiment"] = df_negationReviews["sentiment"].replace("neutral","negation")
df_negationReviews = df_negationReviews.drop(columns ='star_rating') 
df_negationReviews.to_csv("jupyter_notebook_presentation/jupyter_notebook_phase_2/file_appending_method_demo/negation.csv")

df_negation = df_negation.drop(df_negationReviews.index)
df_negation.to_csv("jupyter_notebook_presentation/jupyter_notebook_phase_2/bigger_dataset_after_removal_jupyter/DatasetafterPosNegRm.csv")


df_negative = pd.read_csv("jupyter_notebook_presentation/jupyter_notebook_phase_2/bigger_dataset_after_removal_jupyter/DatasetafterPosNegRm.csv", index_col=0)
df_negativeReviews = df_negative.copy()

negativesummaryPatterns = "horrible|bad|hate|waste|broken|rubbish|don't|doesn't"

df_negativeReviews = df_negativeReviews.loc[(df_negativeReviews['star_rating'].isin([1,2])) & (df_negativeReviews['review_body'].str.contains(negativesummaryPatterns,na= False, case=False))].head(1500) 
df_negativeReviews = df_negativeReviews.drop(columns='star_rating')
df_negativeReviews.to_csv("jupyter_notebook_presentation/jupyter_notebook_phase_2/file_appending_method_demo/negative.csv")

df_negative = df_negative.drop(df_negativeReviews.index)
df_negative.to_csv("jupyter_notebook_presentation/jupyter_notebook_phase_2/bigger_dataset_after_removal_jupyter/DatasetafterPosNegNegativeRm.csv")

After the following method we will have a file with the three appended Categories positive, negation and negative.
All files in the folder "file_appending_methode_demo will be appended. 
In the main system there will be all classes appended.

In [ ]:
import pandas as pd
import os
import glob

path = './jupyter_notebook_presentation/jupyter_notebook_phase_2/file_appending_method_demo/'
all_files = glob.glob(os.path.join(path, "*csv"))

data_frames =[]
total_rows_files = 0 

print("Length of the Data_files")

for f in all_files: 
    temp_df = pd.read_csv(f)
    
    current_rows = temp_df.shape[0]
    total_rows_files += current_rows
    
    print("File:{0} | Rows:{1}".format(os.path.basename(f),current_rows))
    
    data_frames.append(temp_df)
    
df_all_data = pd.concat(data_frames, ignore_index = True)


df_all_data.to_csv("./jupyter_notebook_presentation/jupyter_notebook_phase_2/appended_files/allfiles.csv", index = False)

Length of the Data_files
File:positive.csv | Rows:1500
File:negative.csv | Rows:1500
File:negation.csv | Rows:1000



The last step to clean our data is to reduce the noise in the review_body column, because this column is the main training data for the BERT system. Also the split of the dataset for the training, validation and testing is done here. For this we use the class "text_cleaning_tokenization_and_splitting" 

In [48]:
import re
import janitor_rs
import demoji
from transformers import BertTokenizer
from sklearn.model_selection import train_test_split
from data_classes import text_cleaning_tokenization_and_splitting

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = demoji.replace(text, "")
    text = re.sub(r'[a-zA-Z\s]]', '', text)
    return text.lower().strip()

df_cleaned = (
    pd.read_csv("./jupyter_notebook_presentation/jupyter_notebook_phase_2/appended_files/allfiles.csv")
    .clean_names()
    .remove_empty()
    .transform_column("review_body", clean_text, elementwise =True)
)




Saving the compleptly cleaned dataframe as a csv file in a separate folder 

In [49]:
df_cleaned.to_csv("jupyter_notebook_presentation/jupyter_notebook_phase_2/cleaned_data_jupyter/cleaned_data.csv", index = False)

The next step is to split the training data (questions), that is represented by the review_body Data and the answer data (sentiment) this way the data is neatly organized
This extra step is done to make sure we will not have any data leakage and that the review body will under no circumstances hold the label value this is a extra security step.
It is adding complexity but also insures us, that ther will be no labels in the question data. 

In [50]:
X_all_data_questions = df_cleaned[["id","review_body"]]
y_all_data_answers = df_cleaned["sentiment"]

display(X_all_data_questions.head(3))
display(y_all_data_answers.head(3))

,id,review_body
0,48841,purchased several versions of turbo tax. grea...
1,52972,i was told that the mac version of quickbooks ...
2,28731,the book helped greatly with the side quest'...


0    positive
1    positive
2    positive
Name: sentiment, dtype: str

Because we have 3 datsets the split must be done in 2 parts the methode that is the industry standard to split datasets is just able to split the data in 2 sets at once. 

In [52]:
X_TrainQuestion, X_tempQuestions, y_TrainAnswers, y_tempAnswers = train_test_split (
    X_all_data_questions, y_all_data_answers, 
    test_size = 0.3, # A 70/15/15 Split is the Goal, We first Split the Train and Rest Size 
    random_state= 42,
    stratify= y_all_data_answers  
)
display(X_TrainQuestion.head(3))
display(X_tempQuestions.head(3))
display(y_TrainAnswers.head(3))
display(y_tempAnswers.head(3))


,id,review_body
1175,86341,"as usual, turbo tax meets all my tax preparati..."
2194,109919,don't buy this product if you are expecting th...
389,23430,great educational software


,id,review_body
1646,63087,i should have ordered this from the manufactur...
2439,30862,i got infected even using one. i don't care f...
3609,115682,"i was unable to print high-res tiff files, onl..."


1175    positive
2194    negative
389     positive
Name: sentiment, dtype: str

1646    negative
2439    negative
3609    negation
Name: sentiment, dtype: str

To assign the answer with the question after again we assign the label id to the answer id, this way both have the same id

In [53]:
y_train_answers_with_ID = pd.DataFrame({
    "id" : X_TrainQuestion ["id"],
    "sentiment" : y_TrainAnswers
})

display(y_train_answers_with_ID.head(3))

,id,sentiment
1175,86341,positive
2194,109919,negative
389,23430,positive


To finally use our data for thr 3 Phases we need to tokenize the review body, a machine can just understand numbers, so the tokenizing will transform the strings to numbers. 
After that we save the csv Version to check the data and we create a parquet version to use these files for the training of our BERT model. 
Also the version of the BERT tokenizer needs to be specified.
The label_dict is assigning a string values to the sentiment data, this way its easier to read afterwards and in the csv-file's.

In [55]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize_function(text):
    return tokenizer.encode(text, add_special_tokens=True, max_length=64, truncation=True, padding='max_length')


#label dictonary for the answers 
label_dict ={'negative' :0, 'neutral' :1, 'positive' :2, 'negation':3, 'multipolarity':4, 'sarcastic':5, 'irony':6}


X_TrainQuestion['tokenized_text'] = X_TrainQuestion['review_body'].apply(tokenize_function)
y_train_answers_with_ID['sentiment'] = y_train_answers_with_ID['sentiment'].map(label_dict)

display(X_TrainQuestion.head(3))
display(y_train_answers_with_ID.head(3))

X_TrainQuestion.to_csv("final_datasets/splitted_Datasets/training_Dataset/checktrainingQuestionFormat.csv", index = False)
y_train_answers_with_ID.to_csv("final_datasets/splitted_Datasets/training_Dataset/checktrainingAnswerFormat.csv", index = False)


X_TrainQuestion.to_parquet("./final_datasets/splitted_Datasets/training_Dataset/TrainingQuestions.parquet", index=False)
y_train_answers_with_ID.to_parquet("./final_datasets/splitted_Datasets/training_Dataset/TrainingAnswers.parquet",index=False)





,id,review_body,tokenized_text
1175,86341,"as usual, turbo tax meets all my tax preparati...","[101, 2004, 5156, 1010, 15386, 4171, 6010, 203..."
2194,109919,don't buy this product if you are expecting th...,"[101, 2123, 1005, 1056, 4965, 2023, 4031, 2065..."
389,23430,great educational software,"[101, 2307, 4547, 4007, 102, 0, 0, 0, 0, 0, 0,..."


,id,sentiment
1175,86341,NaN
2194,109919,NaN
389,23430,NaN


Phase 3 the training phase of our BERT system starts with the decision which pretrained base-model we are going to use to train our model. In our case we choose the bert-base-uncased model. num_labels = 7 decides that we will have 7 different classification categories. 

From this point on i dont recommend executing the jupyter notebook code, as its mostly training of the BERT system. The Models already in place should not be retrained. The code added is more for explanatory reasons. 

In [ ]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(       
    'bert-base-uncased',
    num_labels = 7,
    output_hidden_states = False, 
    output_attentions = False
)

Next is the loading of the mps, this is the gpu in the Macbook m1 , training of a Model is mostly done with the gpu thats why we check its avaiability and assign the device to the mps.
Also the model we choose will be send to the gpu for the training. 
The if __name__ == "__main__" is to make sure the classes use their own gpu assignments. 

In [ ]:
import torch
import numpy as np

if __name__ == "__main__": 

 if torch.backends.mps.is_available(): 
    device = torch.device("mps")
    print("Uses Apple M1 8-core gpu for evaluation Dataset")
 else:
    device = torch.device("cpu")
    print("just uses cpu for evaluation")
    
model.to(device)

These steps are neccesary to prepare the training data before we can sart with the training of our model, also the data needs to be transformed into a TensorDataset and we need to decide the parameters for the batch_loader and the parameters of the training of our model. 
df_train_inputs is the parquet file with the id's and the review_body and df_train_labels is the parquet file with the sentiment labels and id's. Firstly we want to make sure the id's are the same in the different datasets so we can be sure there is no missing data. 
In the next step we create the TensorDataset with the torch library and the tensor function. The review body gets transformed from a string to a number, this is the only way how a model can learn it needs the data in number form. Also the labels and the review id's are getting transformed. the attention mask is treating every input that is zero as it is non existing. 
We decide how big the batch size of the train_loader is, 16 is a good size. 
The parameter of the optimizer are deciding which learning rate (lr = ) is used, the eps is a value that we need to set as a safety net, so that a division with 0 is never possible, when the moving average of the gradient gets calculated. It is set at 1e-8, so its small enough to not influence the training. The parameter weight_decay is useful for balancing weights, it is reducing the weights that are getting to big in the model, that way a model is less prone for overfitting.
These parameters are adjustable, in my project i trained 6 different models to see which parameters are delivering the best results, some  models are trained with weight_decay some are without, some have a higher learning rate some a lower rate, the parameter eps was always set to the same value.
Epochs are deciding how many training rounds are done. In most models i just used 3 rounds. 
The last optimization step we can use is the warmup steps, in this case the first 150 batches will use a slower learning rate, this can prevent the model from overfitting. This is especially useful when applying a higher learning rate or more epochs like in this specific case. 
The Warmup steps shoul be beetween 5-10% of the total learning volume.    

In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset
import torch.optim as optimizer
from transformers import get_linear_schedule_with_warmup

df_train_inputs = pd.read_parquet("./final_datasets/splitted_Datasets/training_Dataset/TrainingQuestions.parquet")
df_train_labels = pd.read_parquet("./final_datasets/splitted_Datasets/training_Dataset/TrainingAnswers.parquet")

ids_review_text = df_train_inputs['id'].values
ids_labels = df_train_labels['id'].values

if not (ids_review_text == ids_labels).all():
    raise ValueError("Warning: Id's in review_body and sentiment are not alligned")


input_ids = torch.tensor(df_train_inputs['tokenized_text'].to_list())
attention_mask = (input_ids !=0).long()                         
labels = torch.tensor(df_train_labels['sentiment'].to_list())
row_id_tensor = torch.tensor(ids_review_text).long()

dataset_training = TensorDataset(
input_ids,   
attention_mask, 
labels,  
row_id_tensor 
)

train_loader = DataLoader(dataset_training, batch_size=16, shuffle = True) 

optimizer = optimizer.AdamW(model.parameters(), lr=2e-5, eps =1e-8, weight_decay=0.01)  

epochs = 5
total_training_volume = len(train_loader)* epochs

scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=150, num_training_steps = total_training_volume)


We set the model in train mode with the function train()
to(self.device) is sending the neccesary data to the gpu of my macbook M1. With this device it's actually not really necessary to send the data explicitly to gpu because it uses unified memory(cpu and gpu share the same memory space), but the code is more to show how the traditional training process would look like.
model.zero_grad() resets the gradient weights after every batch, this ensures the loss rate is reseted after every batch and not accumulated. 
outputs will return the loss of the current batch we save the current batch loss in the loss variable. The complete loss of all batches will be saved in the total_loss variable. With loss.item we just extract the pure number and not the whole tensor, this is good for the ram management. The variable loss is still connected to the tensors we need to have it connected, because we going to calculate the gradients of this with the function backward(). This will calculate how wrong the prediction of the model was from the real result. 
optimizer.step() is used afterwards to really adjust the weights and scheduler.step() is used to slightly adjust the learning rate. 
The average train loss for every batch over the whole epoch will be calculated and returned for each epoch. 

In [ ]:
model.train()
for epoch in range(epochs):
    print(f"Start epoch {epoch +1} from {epochs}")
    total_loss = 0

    for batch in train_loader:               
     input_ids = batch[0].to(device)         
     attention_mask = batch[1].to(device)    
     labels = batch[2].to(device)             
     
     model.zero_grad()                       
     
     outputs = model(input_ids,
                     token_type_ids=None,
                     attention_mask=attention_mask,
                     labels=labels)
     
     loss = outputs.loss
     total_loss += loss.item()              
     
     loss.backward()                         
     
     optimizer.step()   
     scheduler.step() 
    average_train_loss = total_loss /len(train_loader) 
    print(f"Average loss in epoch {epoch +1}: {average_train_loss:.4f}")



In the last part of the bert_model_train class we will make sure the models is saved in a folder with all the necessary components to re-use. 
If the path doesnt already exist we create it.
In the package we save the model weights, the config.json file, the used tokenizer(vocabular) and the used model. 
if hasattr is used as a safeguard if the the model was trained on multiple gpu's to make sure we choosing to save the modul where the model is saved. 
We save all of these configurations and files in the same folder so it will be easier to reuse the model. 

In [ ]:
VERSION = "v6_lr2e-5_150warmupsteps_decay0.01_epochs5"  
MODEL_SAVE_PATH = f"./saved_models/{VERSION}"

if not os.path.exists(MODEL_SAVE_PATH):
    os.makedirs(MODEL_SAVE_PATH)

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

model_to_save = model.module if hasattr(model, 'module') else  model 
model_to_save.save_pretrained(MODEL_SAVE_PATH)
tokenizer.save_pretrained(MODEL_SAVE_PATH)

After the training of the model we need to evaluate how good the model performs with never seen data, for this we create the class bert_model_evaluation.
To create a BertEvaluation objekt we need to pass the parameters for the model we use and the device we will use. 
The object will be set immediatly in evaluation mode with self.model.eval() this is very important because we just want to evaluate our data and we dont want to train our model further. 

In [ ]:
class BertEvaluation:
    def __init__(self, model_path , device):
        self.device = device
        self.model = BertForSequenceClassification.from_pretrained((model_path))
        self.model.to(self.device)
        self.model.eval()

Firstly we create some arrays to save the evaluation data. 

The evaluate method needs a dataloader so it can evaluate the data in batches.

Most importantly the batch evaluation needs torch.no_grad() this disables the gradient calculation and reduces memory consumption for computation. 

The parameters input_ids are the reviews itself in tokenized form. input_mask is the relevant of the review_body for the model. 
labels is the sentiment, so which review_body is which customer sentiment. row_ids are the identifier. 
With the variable outputs we use some methods. logits() are the prediction scores of the model. detach() is separating the predicition value from the calculating graph of the model. cpu() is "sending" the data back to the cpu. 
numpy() is putting the values in a standard numpy() Array. 
batch predictions = np.argmax(logits, axis =1).flatten() is choosing the category that has the highest prediction value out of the 7 categories. 

prediction, labels and the ids are extended and collected. 
the accuracy score is getting calculated
After the core calculation we are assigning the values to a dataframe for further operations. 

THe method will return the accuracy , report and the df_validation. 

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

def evaluate(self, dataloader):
        predictions, true_labels, all_row_ids = [],[],[]
        
        print("Start of the evaluation ")
        
        with torch.no_grad():
            for batch in dataloader:
                input_ids = batch[0].to(self.device)
                input_mask = batch[1].to(self.device)
                labels = batch[2].to(self.device)
                row_ids = batch[3] 
                
                outputs = self.model(input_ids, token_type_ids = None, attention_mask = input_mask)
                
                logits = outputs.logits.detach().cpu().numpy()
                labels_ids = labels.to("cpu").numpy()
                
                batch_predictions = np.argmax(logits, axis = 1).flatten() 
                
                predictions.extend(batch_predictions)          
                true_labels.extend(labels_ids)
                all_row_ids.extend(row_ids.cpu().numpy().tolist()) 
                
            accuracy = accuracy_score(true_labels, predictions)
            
            
            df_validation = pd.DataFrame({
                'id': all_row_ids,
                'true_label' : true_labels,
                'prediction_bert':predictions
            })
            
            
            report = classification_report(df_validation['true_label'],df_validation['prediction_bert'])
        
            
            return accuracy, report, df_validation

The performance of the model needs to be saved 
We save the key values as overall accuracy and the classification report
Also we will save the df_validation in a csv-File

In [ ]:
import json

def save_model_performance(self,save_path, accuracy,report,df_validation):
     
     results_summary ={
         "overall_accuracy" : accuracy,
         "classification_report" : report
     }
     
     with open(os.path.join(save_path, "evaluation_metrics_model.json"),"w") as evaluate_json:
         json.dump(results_summary, evaluate_json , indent=4)
     
     df_validation.to_csv(os.path.join(save_path, "model_performance.csv"), index=False)

model_pretrained will be the variable where we going to save the Model, that we are going to use
The df_val_inputs is used for the parquet file that holds the review_body, so the questions.
The df_val_labels is used to hold the answers to the review_body.
Most importantly is that we use different data from the training dataset , the model has never seen this data. 
Next we create the order of the arguments of the TensorDataset. Firstly we rename the column to tokenized_text, where we will have the input id's(review_body) in tokenized form. 
Secondly we asign that just input id's different from 0 are important. labels will be assigned to the column sentiment, and lastly we assign the row id's to the id values. 
The Arguments of the TensorDataset will be put in order and arguments of the DataLoader(batchloader) will be assigned. 
model_pretrained will be finally trained after we assigned all the necessary data , like the tensor Dataset and the batchLoader. The method evaluate is used with the batch_loader as its argument. 
The trained model is saved after training with the method save_model_performance

In [ ]:
from system_classes.bert_model_evaluation import BertEvaluation
from torch.utils.data import TensorDataset, DataLoader

if __name__ == "__main__":

 MODEL_PATH = "v6_lr2e-5_150warmupsteps_decay0.01_epochs5"   

 full_model_path = f"./saved_models/{MODEL_PATH}"      

 model_pretrained = BertEvaluation(full_model_path,device)
 
 df_val_inputs = pd.read_parquet("./final_datasets/splitted_Datasets/validation_Datasets/ValidateQuestions.parquet")
 df_val_labels = pd.read_parquet("./final_datasets/splitted_Datasets/validation_Datasets/ValidateAnswers.parquet")

 input_ids = torch.tensor(df_val_inputs['tokenized_text'].to_list())
 attention_mask = (input_ids != 0).long()
 labels = torch.tensor(df_val_labels['sentiment'].to_list())
 row_ids = torch.tensor(df_val_inputs['id'].values).long()
                
 val_dataset = TensorDataset(input_ids, attention_mask, labels, row_ids)
 val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
 accuracy , report, df_validation = model_pretrained.evaluate(val_loader)
 
 MODEL_SAVE_PATH = f"./saved_models/{MODEL_PATH}"
 
 model_pretrained.save_model_performance( MODEL_SAVE_PATH, accuracy, report, df_validation)

For the Bert Test Class we use pretty much the same method but of course we use a different DataSet, the results did not differ a lot from the results of the Evaluation Dataset. 

In [ ]:
import torch
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset
from system_classes.bert_model_evaluation import BertEvaluation 

if torch.backends.mps.is_available():  # new instance 
    device = torch.device("mps")
    print("Uses Apple M1 8-core gpu for test Dataset")
else:
    device = torch.device("cpu")
    print("just uses cpu for evaluation")


MODEL_PATH = "v6_lr2e-5_150warmupsteps_decay0.01_epochs5"  
       
evaluator_test_phase = BertEvaluation(f"saved_models/{MODEL_PATH}", device)    

df_test_input = pd.read_parquet("./final_datasets/splitted_Datasets/test_Dataset/TestQuestion.parquet")
df_test_labels = pd.read_parquet("final_datasets/splitted_Datasets/test_Dataset/TestAnswers.parquet")

input_ids = torch.tensor(df_test_input['tokenized_text'].to_list())
attention_mask = (input_ids != 0).long()
labels = torch.tensor(df_test_labels['sentiment'].to_list())
row_ids = torch.tensor(df_test_input['id'].values).long()

test_dataset =  TensorDataset(input_ids, attention_mask, labels, row_ids)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)
    
accuracy, report, df_test_result = evaluator_test_phase.evaluate(test_loader)
    
print(f"Accuracy test dataset :{accuracy}")
print(report)

df_test_result.to_csv("./final_datasets/splitted_Datasets/test_Dataset/test_results.csv",index=False) 

The calculate_weighted_sentiment Class was created to combine different methods for text classification, originally the plan was to combine the Vader, Textblob and Bert System. But after some experimantation with the System, i decided to just use the Bert System for the final Judgement which category it really is. The Vader and TextBlob Systems are just to basic for most complex categories like multipolarity, negation, irony and sarcasm. The basic categories get influenced in the final prediction somewhat by these systems but bert alone is also more reliable. 

In the case of a complex category review bert decides alone which category it could be
In the case of a non-complex case a weighted score is getting calculated ,bert has the most weight. 

In [ ]:
def calculate_weighted_sentiment(vader_score, textBlob_score, bert_class ):
    
    complex_mapping ={
    3: "negation",
    4: "multipolarity",
    5: "sarcasm",
    6: "irony"
    }
    
    if bert_class in complex_mapping:
        return complex_mapping[bert_class]
    
    if bert_class == 3:
        return "negation"
    elif bert_class == 4:
        return "multipolarity"
    elif bert_class == 5:
        return "sarcasm"
    elif bert_class == 6:
        return "irony"
    
    bert_score =(bert_class-3)/3.0     
    
    weight_textblob = 1.0
    weight_vader = 1.5
    weight_bert = 2.5
    
    total_weight = weight_textblob + weight_vader + weight_bert
        
    weighted_score =((textBlob_score * weight_textblob) +
                     (vader_score * weight_vader)+
                     (bert_score * weight_bert))  / total_weight
     
     
    if weighted_score >=0.05:
        return "positive", weighted_score
    if weighted_score <= -0.05:
        return "negative", weighted_score
    else:
        return "neutral", weighted_score

Firstly we collect all csv's and merge them. Some adjustments were neccesary becuase of the merge and duplicates. 
Than we applying all the scores from each system to the dataframe df_final 
The results of the 3 systems combined are assigned to hybrid_label column that reflects the sentiment category and to the column hybrid_score that reflects a numeric score. 
hybrid_label for complex categories is always the sentiment of bert alone, hybrid_label of normal categories is the real mixed sentiments of all 3 systems. 

In [ ]:
df_textblob_vader= pd.read_csv("./final_datasets/dataset_weighted_scores/weighted_scores.csv")

df_bert = pd.read_csv("final_datasets/splitted_Datasets/test_Dataset/test_results.csv")

if 'prediction' in df_textblob_vader.columns:
    df_textblob_vader = df_textblob_vader.drop(columns=['prediction']) # because of the merge method this is integrated, wo stop merge from creating new prediction columns
    
#df_textblob_vader['id'] = df_textblob_vader['id'].astype(int)
#df_bert['id'] = df_bert['id'].astype(int)

df_final = df_textblob_vader.merge(df_bert[['id', 'prediction']], on='id', how='inner')
df_final = df_final.drop_duplicates(subset=['id'], keep='first') # some reviews had duplicates after merge, removing them 


if 'hybrid_label' in df_final.columns:
    df_final = df_final.drop(columns=['hybrid_label', 'hybrid_score']) 

results = df_final.apply(
    lambda x: calculate_weighted_sentiment(
        vader_score =x['vader_score'],
        textBlob_score=x['textblobscore'],
        bert_class = x['prediction']
        ), 
    axis=1
)

df_final['hybrid_label'] = results.apply(lambda x: x[0] if isinstance(x, (tuple, list)) else x)
df_final['hybrid_score'] = results.apply(lambda x: x[1] if isinstance(x, (tuple, list)) else 0.0)

df_final[['weighted_score']] = df_final[['weighted_score']].round(3)
df_final[['hybrid_score']] = df_final[['hybrid_score']].round(3)

A lot of experimenting how well each system performs and how they perform together

In [ ]:
sentiment_prediction_maping_numeric_to_string = {
    0: "negative",
    1: "neutral",
    2: "positive", 
    3: "negation",
    4: "multipolarity",
    5: "sarcastic", 
    6: "irony"
}

df_prediction_numeric_to_string = df_final.copy()
df_prediction_numeric_to_string['prediction'] =  df_prediction_numeric_to_string['prediction'].map(sentiment_prediction_maping_numeric_to_string)

accuracy_vader_textBlob_all_Categories = accuracy_score( df_final['sentiment'],df_final['weighted_sentiment_label'])
accuracy_all_systems_allCategories = accuracy_score(df_final['sentiment'], df_final['hybrid_label'])
accuracy_just_Bert_all_Categories = accuracy_score(df_prediction_numeric_to_string['sentiment'], df_prediction_numeric_to_string['prediction']) 


print(f"Accuracy VADER + TextBlob all categories: {accuracy_vader_textBlob_all_Categories:.4f}")
print(f"Accuracy Vader + TextBlob +Bert all categories : {accuracy_all_systems_allCategories:.4f}")
print(f"Accuracy just Bert all categories: {accuracy_just_Bert_all_Categories:.4f}")


#Spltting the Data in base Categories and Complex Categories for further insights
target_labels = ["positive", "neutral", "negative"]
df_filtered_base_categories = df_final[df_final['sentiment'].isin(target_labels)].copy()

target_labels_complex = ["negation", "multipolarity", "sarcastic", "irony"]
df_filtered_complex_categories = df_final[df_final['sentiment'].isin(target_labels_complex)].copy()

#maping both Dataframes base and complex, this way we can use the prediction column to track the performance of the bert model and to compare performances between bert and vader/TextBlob
df_prediction_mapped_base_category = df_filtered_base_categories.copy()
df_prediction_mapped_base_category['prediction'] = df_prediction_mapped_base_category['prediction'].map(sentiment_prediction_maping_numeric_to_string)

df_predcition_mapped_complex_categories = df_filtered_complex_categories.copy()
df_predcition_mapped_complex_categories['prediction'] = df_predcition_mapped_complex_categories['prediction'].map(sentiment_prediction_maping_numeric_to_string)

accuracy_base_categories_vader_textBlob = accuracy_score(df_filtered_base_categories['sentiment'],df_filtered_base_categories['weighted_sentiment_label'])
accuracy_base_categories_allSystems = accuracy_score(df_filtered_base_categories['sentiment'],df_filtered_base_categories['hybrid_label'])
accuracy_base_categories_justBert = accuracy_score(df_prediction_mapped_base_category['sentiment'],df_prediction_mapped_base_category['prediction'])
accuracy_complex_categories_justBert = accuracy_score(df_predcition_mapped_complex_categories['sentiment'], df_predcition_mapped_complex_categories['prediction'])


print(f"Accuracy VADER + TextBlob base categories(positive, neutral, negative): {accuracy_base_categories_vader_textBlob:.4f}")
print(f"Accuracy Vader + TextBlob +  BERT base categories(positive, neutral, negative): {accuracy_base_categories_allSystems:.4f}")
print(f"Accuracy base categories just Bert (positive, neutral, negative): {accuracy_base_categories_justBert:.4f}")
print(f"Accuracy complex categories Bert (negation,multipolarity,sarcastic,irony): {accuracy_complex_categories_justBert:.4f}")

df_final.to_csv("./final_datasets/dataset_weighted_scores/weighted_scores.csv", index= False)
df_filtered_base_categories.to_csv("./final_datasets/dataset_weighted_scores/base_categories.csv", index =False)
df_prediction_mapped_base_category.to_csv("./final_datasets/dataset_weighted_scores/bert_base_categories.csv", index =False)
df_predcition_mapped_complex_categories.to_csv("./final_datasets/dataset_weighted_scores/bert_complex_categories.csv", index = False)

# Creating a Dataframe where all predictions are converted to String
# preparing the Data for the csv Export 


df_tableu_data = df_prediction_numeric_to_string
df_tableu_data['correct_bert'] = (df_tableu_data['sentiment'] == df_tableu_data['prediction'])
df_tableu_data['correct_vader_and_textblob'] = (df_tableu_data['sentiment']== df_tableu_data['weighted_sentiment_label'])

# creating a Method so we have a comparison which model was right for the prediction of the sentiment 

def match_data(row):
    if row['sentiment'] == row['prediction'] and row['sentiment']== row['weighted_sentiment_label']:
       return "Both Systems are correct"
    elif row ['sentiment'] == row['prediction']:
        return "Bert Correct"
    elif row ['sentiment'] == row ['weighted_sentiment_label'] :
        return "Vader and Textblob Correct"
    else: 
        return "Both Systems are Wrong"

Preparing the dataframe for the export to tableu
Renaming the prediction column for easier identification in tableu 
Saving the csv in the folder tableu data 

In [ ]:
df_tableu_data['wright_system'] = df_tableu_data.apply(match_data,axis=1)


df_tableu_data = df_tableu_data.rename(columns={"prediction" : "prediction_bert", "sentiment": "true_sentiment"})

df_tableu_data.to_csv("./final_datasets/tableu_data/tableu_export_v2.csv", index= False)

Now we moving into Phase 4 the Class marketing_datsets_input_User_Interface is designed to open the finder window and to let the employee of the marketing department choosing a parquet file.
This Class is designed to evaluate the parquet file that was choosen from the employee. In most real life use cases raw Data is avaiable from a certain source. The only constraint we have here is that we need a parquet file and that it needs to have some certain columns with specific names. 

After starting the class the employee can choose a file. If the choosen file's file path ends with parquet, we transform the parquet file into a dataframe. Also we get a short feedback that the file was succesfully loaded. 

The evaluate marketing data is almost the same as the normal evaluate method but we jst have 2 arrays for the values of the bert predictions and the row_ids

After the evaluation process we assign the values prediciton_bert and id to the correponding columns and we also add the matching review text to the datframe with the id. The decision to also add the review text was done to create a better visualization in tableu, which review holds which review text?

In [ ]:
import torch
import numpy as np
import pandas as pd
from transformers import BertForSequenceClassification
import tkinter as tk
from tkinter import filedialog
from torch.utils.data import TensorDataset, DataLoader
import datetime
from datetime import datetime


if torch.backends.mps.is_available(): 
    device = torch.device("mps")
    print("Uses Apple M1 8-core gpu for evaluation Dataset")
else:
    device = torch.device("cpu")
    print("just uses cpu for evaluation")

class MarketingEvaluation:
    def __init__(self,model_path, device ):
     self.device = device
     self.model = BertForSequenceClassification.from_pretrained((model_path))
     self.model.to(self.device)
     self.model.eval()
     self.input_path = None

     
    def file_selector(self):
        root = tk.Tk()
        root.withdraw()
        root.attributes('-topmost', True)
        
        file_path = filedialog.askopenfilename(
        title = "Choose the file with the customer review data, just parquet files are allowed",
        filetypes=[("Allowed data format", "*.parquet")]
        )
        
        root.destroy()
        
        self.input_path = file_path
        
        if file_path.endswith('parquet'):
            self.df_marketing_predictions = pd.read_parquet(file_path)
            
        print(f"File was loaded{len(self.df_marketing_predictions)} reviews found")
        
        return self.df_marketing_predictions
     
    
    
    def evaluate_marketing_data(self, dataloader):
        predictions_bert, all_row_ids = [],[]
        
        print("Start of the Customer Review analysis")
        
        with torch.no_grad():
            for batch in dataloader:
                input_ids = batch[0].to(self.device)
                input_mask = batch[1].to(self.device)
                row_ids = batch[2] 
                
                outputs = self.model(input_ids, token_type_ids = None, attention_mask = input_mask)
                
                logits = outputs.logits.detach().cpu().numpy()
                batch_predictions = np.argmax(logits, axis = 1).flatten()  
                
                predictions_bert.extend(batch_predictions)          
                all_row_ids.extend(row_ids.cpu().numpy().tolist())  
                
            df_marketing_predictions_bert = pd.DataFrame({
                'id': all_row_ids,
                'prediction_bert':predictions_bert
            })
            
            sentiment_prediction_maping_numeric_to_string = {
            0: "negative",
            1: "neutral",
            2: "positive", 
            3: "negation",
            4: "multipolarity",
            5: "sarcastic", 
            6: "irony"
            }
            
            df_marketing_predictions_bert['prediction_bert'] = df_marketing_predictions_bert['prediction_bert'].map(sentiment_prediction_maping_numeric_to_string)
            
            df_marketing_with_review_text = pd.merge(
                df_marketing_predictions_bert,
                self.df_marketing_predictions [['id', 'review_body']],
                on='id',
                how='left'
            )
                
            return  df_marketing_with_review_text

After the creation of the dataset we also need to save them, the system is that we always want to keep the older data in a Folder called archive_data , at the same time we want to deliver the new data to tableu for this case we have the folder live_data , the path stays the same so the newer csv will overwrite the old one, the tableu visualization will show the new data. For better overview we are adding a timestamp to each csv-file. 

In [ ]:
def save_customer_review_predciton(self,df_marketing_with_review_text):
        
        timestamp = datetime.now().strftime(f"%Y-%m-%d %H:%M:%S")
        
        archive_path = f"./final_datasets/tableu_data/archive_data/tableu_export_{timestamp}.csv"
        df_marketing_with_review_text.to_csv(archive_path, index=False)
        
        live_path = "./final_datasets/tableu_data/live_data/marketing_current_data.csv"
        df_marketing_with_review_text.to_csv(live_path, index=False)

In the last part of the marketing_datasets_input_User_interface class we apply the methods to our data. We also choose the best fitting model for daily use i choodes the most realistic model that is least prone to overfitting. These Parameters will be choosen by the IT-Department the Marketing departments concern is just to feed the interface new parquet files for new data. 

For Testing the System i recommend using the already avaiable datasets in the folder splitted_Datasets : 
- final_datasets/splitted_Datasets/test_Dataset/TestQuestion.parquet
- final_datasets/splitted_Datasets/validation_Datasets/ValidateQuestions.parquet
These datasets have the format and file type for the tests. 
Testing should be done in an IDE by running the class marketing_datasets_input_User_interface.py

In [ ]:
MODEL_FIT = "v3_lr2e-5_100warmupsteps_0.01weight_decay"
MODEL_USED = f"./saved_models/{MODEL_FIT}"

marketing_input_and_model = MarketingEvaluation(MODEL_USED,device)

file_input = marketing_input_and_model.file_selector()

input_ids = torch.tensor(file_input['tokenized_text'].to_list())
attention_mask = (input_ids != 0).long()
row_ids = torch.tensor(file_input['id'].values).long()

customer_reviews = TensorDataset(input_ids, attention_mask, row_ids)
customer_reviews_dataloader = DataLoader(customer_reviews, batch_size=16, shuffle= False)

df_customer_review_for_tableu = marketing_input_and_model.evaluate_marketing_data(customer_reviews_dataloader)

marketing_input_and_model.save_customer_review_predciton(df_customer_review_for_tableu)

After executing the class the csv files will be created in the folders, the live folder is connected to tableu, this way the data is automatically updated. 
For better visualization of the data the presentation mode in tableu should be used 